# Day 65 — Preprocessing: Retail Demand Predictor Mini-Project

This notebook covers Preprocessing Pipeline, One-Hot Encoding Categorical Features, Feature Scaling via StandardScaler, ColumnTransformer Architecture, and Preventing Data Leakage

## Validation goals
- Understand the necessity of data preprocessing for machine learning models requiring clean, numerically encoded, and scaled tensors
- Implement categorical feature encoding using One-Hot Encoding to translate text strings into binary numerical representations
- Apply feature scaling (`StandardScaler`) to normalize numerical variables with disparate magnitudes and units
- Construct a robust Scikit-Learn `ColumnTransformer` pipeline and strictly isolate training splits to prevent data leakage

### Preprocessing.

Raw data is rarely ready to be fed straight into a machine learning model. Machine learning algorithms require clean, numerically encoded, and properly scaled tensors. Let's build a robust preprocessing pipeline from first principles.

Before a model can learn patterns from our retail dataset, we must solve three structural challenges:
- Categorical Encoding: Algorithms like linear regression and tree models cannot compute math on text strings like "Lagos", "Groceries", or "Festive". We must convert these categories into numerical binary representations using One-Hot Encoding.
- Feature Scaling: Our numerical features have drastically different scales. A unit `price_ngn` can be $\\₦75,000$, whereas `vat_rate` is 0.075 and `historical_sales_lag` is 150. Without scaling, algorithms can overvalue features with larger numeric magnitudes simply because of their unit scale.
- Data Splitting: To prevent data leakage, we must strictly isolate our training data from our test data before applying any transformations like scaling.

### 2. Building the Preprocessing Pipeline in Python

Let's write a clean, production-grade preprocessing script using Scikit-learn's ColumnTransformer, StandardScaler, and OneHotEncoder.

(Note: This script assumes you have df_retail initialized from yesterday's session.)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Load the raw dataset saved on Day 64
df_retail = pd.read_csv("retail_demand_predictor.csv")

# 2. Separate Feature Matrix (X) and Target Label (y)
X = df_retail.drop(columns=['demand_units'])
y = df_retail['demand_units']

# 3. Split data FIRST (80% Train / 20% Test) to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 4. Define column groups and build the ColumnTransformer
categorical_cols = ['location', 'category', 'season']
numerical_cols = ['price_ngn', 'vat_rate', 'fuel_transport_cost_ngn', 'historical_sales_lag']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

# 5. Fit on training data, transform both splits
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 6. Reconstruct DataFrames with clean feature names
feature_names = preprocessor.get_feature_names_out()

train_df = pd.DataFrame(X_train_processed, columns=feature_names)
train_df['demand_units'] = y_train.values  # Re-attach target label

test_df = pd.DataFrame(X_test_processed, columns=feature_names)
test_df['demand_units'] = y_test.values    # Re-attach target label

# 7. Save processed datasets to CSV for Day 66+
train_df.to_csv("retail_demand_train_processed.csv", index=False)
test_df.to_csv("retail_demand_test_processed.csv", index=False)

print("Saved 'retail_demand_train_processed.csv' and 'retail_demand_test_processed.csv' successfully.")

Saved 'retail_demand_train_processed.csv' and 'retail_demand_test_processed.csv' successfully.


### Guided Question:

Why is it a critical mistake to fit your scaler or encoder on the entire dataset before splitting it into training and test sets?

### Answer:

Fitting a scaler (e.g., `StandardScaler`, `MinMaxScaler`) or an encoder (e.g., `OneHotEncoder`, `TargetEncoder`) on the entire dataset before splitting it into training and test sets is a critical mistake because it causes data leakage—specifically, the unauthorized transfer of information from the test set into the training process.

**1. The Core Mechanism of Data Leakage**

When you call .`fit()` on a transformer using the entire dataset ($X_{\text{all}}$), the transformer computes global summary statistics across every available sample:
- For Scalers: The global mean ($\mu_{\text{all}}$) and standard deviation ($\sigma_{\text{all}}$) incorporate values from the test set. When you subsequently scale the training set using these global parameters, every training sample is adjusted using information derived partly from the test set.
- For Encoders: Category frequencies, global category mappings, or (in the case of target encoding) target mean statistics are calculated using rows that the model is supposed to treat as strictly "unseen."

**2. Why This Compromises Your Model**

Artificially Inflated Evaluation Metrics
By exposing the training pipeline to the distribution parameters of the test set, your model indirectly "sees" the test data. During evaluation, your test score will look optimistically high because the test features have already been normalized or encoded using their own hidden properties. This creates a false sense of security, masking how poorly the model will actually generalize to truly out-of-sample data.

**Production Deployment Failure**

In a real-world production environment, your model will encounter inference data one batch at a time—data that has no pre-calculated global mean or variance. If your training pipeline relied on statistics from a historical test split that no longer exists, you cannot replicate that transformation in production. You would have to scale inference data using training-only statistics, resulting in a distribution mismatch that severely degrades model accuracy.

**The Correct Workflow**

To prevent leakage, the transformation pipeline must treat the test set as strictly out-of-sample data.
- Split First: Partition your raw data into training and test sets before any preprocessing.
- Fit on Train Only: Learn the scaling or encoding parameters exclusively from the training set (`scaler.fit(X_train)`).
- Transform Both Independently: Apply those learned parameters to transform both the training set and the test set (s`caler.transform(X_train)` and `scaler.transform(X_test)`).

### Guided Question:

In the Scikit-learn pipeline above, why did we use StandardScaler on features like `price_ngn` and `fuel_transport_cost_ngn`?

### Answer:

We used `StandardScaler` on features like `price_ngn` and `fuel_transport_cost_ngn` to normalize their numerical ranges to a mean of $0$ and a standard deviation of $1$.In the context of the retail demand dataset, raw monetary values introduce significant scale discrepancies that directly impact model training and convergence:

**1. Eliminating Feature Dominance in Optimization**

Features like `price_ngn` (often in thousands or millions of Naira) and `fuel_transport_cost_ngn` have massive numerical magnitudes compared to ratios like `vat_rate` (e.g., `0.075`) or small counts like `historical_sales_lag`.

If left unscaled, optimization algorithms driven by gradient descent (such as linear regression with gradient solver, neural networks, or regularized linear models) will treat changes in large-magnitude features as vastly more important than changes in smaller-magnitude features. The loss surface becomes heavily skewed (elongated in the direction of the unscaled features), causing gradient updates to bounce erratically and stalling convergence.

**2. Ensuring Fair Regularization (L1/L2 Penalties)**

If you apply regularized models (like Ridge or Lasso regression to prevent overfitting), the penalty term penalizes the absolute size of the model coefficients ($\sum \beta^2$ or $\sum \vert{}\beta\vert{}$).
- Without scaling, a feature measured in tens of thousands of Naira requires a tiny coefficient to make the same prediction impact as a feature measured in single digits
- Regularization would unfairly penalize features with larger natural scales, assuming they represent greater model complexity simply because of their units of measurement. Scaling puts all features on an equal playing field.

**3. Improving Numerical Stability**

Large raw numerical values combined with complex matrix multiplications can occasionally lead to numerical instability, floating-point precision issues, or overflow/underflow errors during matrix inversions or iterative weight updates. Standardization keeps values tightly bound around zero (typically between $-3$ and $+3$), ensuring stable numerical computation.